For mapping purposes, we could take the average of the child cell elevations, or we could evaluate on min and on max elevation of the children. It's challenging: with the maps, we are making a statement that a taxon does or does not occur in this grid cell.  That probably shouldn't be based on the elevation of the centroid but of the range of elevations that are contained in the centroid, right?

In [58]:
import colorsys
import hashlib
import json
import math
import os
from pathlib import Path

import geopandas as gpd
import h3
import h3pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm.auto import tqdm
import pydeck as pdk

In [70]:
import sys
sys.path.append("../")
from lib.models.geo_model_net import ResLayer
from lib.models.encoders import CoordEncoder

In [9]:
tax_2_23 = pd.read_csv(
    "/data-ssd/alex/datasets/vision-export-20250622060005-aka-2.23/taxonomy.csv"
)

In [10]:
class TFGeoPriorModel:

    def __init__(self, model_path):
        # initialize the geo model for inference
        tf.config.set_visible_devices([], "GPU")
        visible_devices = tf.config.get_visible_devices()
        for device in visible_devices:
            assert device.device_type != "GPU"
        self.gpmodel = tf.keras.models.load_model(
            model_path,
            custom_objects={"ResLayer": ResLayer},
            compile=False
        )

    def predict(self, latitude, longitude):
        encoded_loc = TFGeoPriorModel.encode_loc([latitude], [longitude])
        return self.gpmodel(tf.convert_to_tensor(
            tf.expand_dims(encoded_loc[0], axis=0)
        ), training=False)[0]

    def features_for_one_class(self, latitude, longitude):
        """Evalutes the model for a single class and multiple locations

        Args:
            latitude (list): A list of latitudes
            longitude (list): A list of longitudes (same length as latitude)
            class_of_interest (int): The single class to eval

        Returns:
            numpy array: scores for class of interest at each location
        """

        encoded_loc = TFGeoPriorModel.encode_loc(latitude, longitude)
        loc_emb = self.gpmodel.layers[0](encoded_loc)

        # res layers - feature extraction
        x = self.gpmodel.layers[1](loc_emb)
        x = self.gpmodel.layers[2](x)
        x = self.gpmodel.layers[3](x)
        x = self.gpmodel.layers[4](x)

        return x

    def eval_one_class_from_features(self, features, class_of_interest):
        # process just the one class
        return tf.keras.activations.sigmoid(
            tf.matmul(
                tf.expand_dims(self.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
                features,
                transpose_b=True
            )
        ).numpy()

    @staticmethod
    def encode_loc(latitude, longitude):
        latitude = np.array(latitude)
        longitude = np.array(longitude)
        grid_lon = longitude.astype("float32") / 180.0
        grid_lat = latitude.astype("float32") / 90.0

        norm_loc = tf.stack([grid_lon, grid_lat], axis=1)

        encoded_loc = tf.concat([
            tf.sin(norm_loc * math.pi),
            tf.cos(norm_loc * math.pi),
        ], axis=1)
        return encoded_loc

In [15]:
geo_model_2_23 = TFGeoPriorModel(
    "/home/alex/inatGeoModelTraining/saved_models/saved_model_2_23_coords_elev.h5"
)

In [16]:
geo_model_2_23.gpmodel.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 256)               1536      
                                                                 
 res_layer_4 (ResLayer)      (None, 256)               131584    
                                                                 
 res_layer_5 (ResLayer)      (None, 256)               131584    
                                                                 
 res_layer_6 (ResLayer)      (None, 256)               131584    
                                                                 
 res_layer_7 (ResLayer)      (None, 256)               131584    
                                                                 
 dense_9 (Dense)             (None, 106407)            27240192  
                                                                 
 dropout_4 (Dropout)         (None, 106407)            0

# let's get all cells in california

In [390]:
california_polygon = {
    "type": "Polygon",
    "coordinates": [[
        [-124.482003, 42.000325],
        [-114.131211, 41.995232],
        [-114.465747, 24.542245],
        [-124.409591, 24.534156],
        [-124.482003, 42.000325]
    ]]
}

In [391]:
resolution = 4
h3_indices = h3.polyfill(california_polygon, resolution, geo_json_conformant=True)

In [392]:
df = pd.DataFrame({
    "h3_04": list(h3_indices)
}).set_index("h3_04")

In [393]:
dfh3 = df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [394]:
raster = np.load("/home/alex/sinr/data/env/elev_scaled.npy")
encoder = CoordEncoder(
    encoding_strategy="sinusoidal",
    raster=raster
)

In [395]:
locs = dfh3.to_numpy()
encoded_locs = encoder.encode(locs)

In [396]:
class_of_interest = int(tax_2_23[tax_2_23["name"] == "Crotalus ruber"].iloc[0].leaf_class_id)

In [397]:
threshold = 0.13178508

In [398]:
tax_2_23[tax_2_23["name"] == "Crotalus ruber"]

,parent_taxon_id,taxon_id,rank_level,leaf_class_id,iconic_class_id,spatial_class_id,name,iconic_taxon_id,rank
13779,30692.0,30724,10.0,85573.0,5.0,85573,Crotalus ruber,26036.0,species


In [399]:
loc_emb = geo_model_2_23.gpmodel.layers[0](encoded_locs)

# res layers - feature extraction
x = geo_model_2_23.gpmodel.layers[1](loc_emb)
x = geo_model_2_23.gpmodel.layers[2](x)
x = geo_model_2_23.gpmodel.layers[3](x)
geo_model_feats = geo_model_2_23.gpmodel.layers[4](x)

raw_preds = tf.matmul(
    tf.expand_dims(geo_model_2_23.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
    geo_model_feats,
    transpose_b=True
)
preds = tf.keras.activations.sigmoid(raw_preds).numpy()

In [400]:
dfh3["preds"] = preds[0]

In [401]:
dfh3.reset_index(inplace=True)

In [402]:
dfh3["fill_color"] = dfh3["preds"].apply(lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100])

In [403]:
dfh3

,h3_04,lng,lat,preds,fill_color
0,8429a41ffffffff,-117.235722,32.826164,0.230034,"[255, 196, 196, 100]"
1,8429813ffffffff,-117.635647,37.656258,0.003756,"[255, 254, 254, 100]"
2,842811dffffffff,-122.016198,41.142055,0.000555,"[255, 254, 254, 100]"
3,8448469ffffffff,-115.080070,27.793561,0.316571,"[255, 174, 174, 100]"
4,8448491ffffffff,-119.550241,31.605962,0.001748,"[255, 254, 254, 100]"
...,...,...,...,...,...
935,8429341ffffffff,-121.733834,29.209427,0.000411,"[255, 254, 254, 100]"
936,845091bffffffff,-122.038530,25.654398,0.000245,"[255, 254, 254, 100]"
937,842926dffffffff,-123.611793,28.774531,0.000239,"[255, 254, 254, 100]"
938,84291adffffffff,-121.719231,35.698649,0.000163,"[255, 254, 254, 100]"


In [404]:
dfh3["preds_thresh"] = dfh3.preds.apply(lambda x: 0.5 if x > threshold else 0)
dfh3["thresh_fill_color"] = dfh3["preds_thresh"].apply(lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100])

In [405]:
dfh3

,h3_04,lng,lat,preds,fill_color,preds_thresh,thresh_fill_color
0,8429a41ffffffff,-117.235722,32.826164,0.230034,"[255, 196, 196, 100]",0.5,"[255, 127, 127, 100]"
1,8429813ffffffff,-117.635647,37.656258,0.003756,"[255, 254, 254, 100]",0.0,"[255, 255, 255, 100]"
2,842811dffffffff,-122.016198,41.142055,0.000555,"[255, 254, 254, 100]",0.0,"[255, 255, 255, 100]"
3,8448469ffffffff,-115.080070,27.793561,0.316571,"[255, 174, 174, 100]",0.5,"[255, 127, 127, 100]"
4,8448491ffffffff,-119.550241,31.605962,0.001748,"[255, 254, 254, 100]",0.0,"[255, 255, 255, 100]"
...,...,...,...,...,...,...,...
935,8429341ffffffff,-121.733834,29.209427,0.000411,"[255, 254, 254, 100]",0.0,"[255, 255, 255, 100]"
936,845091bffffffff,-122.038530,25.654398,0.000245,"[255, 254, 254, 100]",0.0,"[255, 255, 255, 100]"
937,842926dffffffff,-123.611793,28.774531,0.000239,"[255, 254, 254, 100]",0.0,"[255, 255, 255, 100]"
938,84291adffffffff,-121.719231,35.698649,0.000163,"[255, 254, 254, 100]",0.0,"[255, 255, 255, 100]"


In [407]:
view_state_grid = pdk.ViewState(
    latitude=34, 
    longitude=-117, 
    zoom=6, 
    bearing=0, 
    pitch=0
)
layer_grid = pdk.Layer(
    "H3HexagonLayer",
    dfh3.to_dict("records"),
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="thresh_fill_color",
    line_width_min_pixels=0,
)
r_grid = pdk.Deck(
    layers=[layer_grid], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
#r_grid.show()
r_grid.to_html("crotalus_ruber_r4_centroid_thresh.html", notebook_display=False)

# ok let's do the same thing for r5

In [408]:
resolution = 5
h3_indices = h3.polyfill(california_polygon, resolution, geo_json_conformant=True)
df = pd.DataFrame({
    "h3_05": list(h3_indices)
}).set_index("h3_05")
dfh3 = df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [409]:
locs = dfh3.to_numpy()
encoded_locs = encoder.encode(locs)

In [410]:
loc_emb = geo_model_2_23.gpmodel.layers[0](encoded_locs)

# res layers - feature extraction
x = geo_model_2_23.gpmodel.layers[1](loc_emb)
x = geo_model_2_23.gpmodel.layers[2](x)
x = geo_model_2_23.gpmodel.layers[3](x)
geo_model_feats = geo_model_2_23.gpmodel.layers[4](x)

raw_preds = tf.matmul(
    tf.expand_dims(geo_model_2_23.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
    geo_model_feats,
    transpose_b=True
)
preds = tf.keras.activations.sigmoid(raw_preds).numpy()

In [411]:
dfh3["preds"] = preds[0]
dfh3.reset_index(inplace=True)
dfh3["fill_color"] = dfh3["preds"].apply(lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100])

In [412]:
dfh3["preds_thresh"] = dfh3.preds.apply(lambda x: 0.5 if x > threshold else 0)
dfh3["thresh_fill_color"] = dfh3["preds_thresh"].apply(lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100])

In [415]:
view_state_grid = pdk.ViewState(
    latitude=34, 
    longitude=-117, 
    zoom=6, 
    bearing=0, 
    pitch=0
)
layer_grid = pdk.Layer(
    "H3HexagonLayer",
    dfh3.to_dict("records"),
    pickable=True,
    stroked=True,
    filled=True,
    extruded=False,
    get_hexagon="h3_05",
    get_fill_color="thresh_fill_color",
    line_width_min_pixels=0,
)
r_grid = pdk.Deck(
    layers=[layer_grid], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
#r_grid.show()
r_grid.to_html("crotalus_ruber_r6_centroid_thresh.html", notebook_display=False)

# ok let's try bubbling up r5 to r4

In [416]:
resolution = 5
h3_indices = h3.polyfill(california_polygon, resolution, geo_json_conformant=True)
df = pd.DataFrame({
    "h3_05": list(h3_indices)
}).set_index("h3_05")
dfh3 = df.h3.h3_to_geo()
dfh3["lng"] = dfh3.geometry.x
dfh3["lat"] = dfh3.geometry.y
_ = dfh3.pop("geometry")

In [417]:
locs = dfh3.to_numpy()
encoded_locs = encoder.encode(locs)

In [418]:
loc_emb = geo_model_2_23.gpmodel.layers[0](encoded_locs)

# res layers - feature extraction
x = geo_model_2_23.gpmodel.layers[1](loc_emb)
x = geo_model_2_23.gpmodel.layers[2](x)
x = geo_model_2_23.gpmodel.layers[3](x)
geo_model_feats = geo_model_2_23.gpmodel.layers[4](x)

raw_preds = tf.matmul(
    tf.expand_dims(geo_model_2_23.gpmodel.layers[5].weights[0][:, class_of_interest], axis=0),
    geo_model_feats,
    transpose_b=True
)
preds = tf.keras.activations.sigmoid(raw_preds).numpy()

In [419]:
dfh3["preds"] = preds[0]

In [420]:
dfh3 = dfh3.h3.h3_to_parent_aggregate(4, operation='max', return_geometry=False)

In [421]:
dfh3.reset_index(inplace=True)
dfh3["fill_color"] = dfh3["preds"].apply(lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100])

In [422]:
dfh3["preds_thresh"] = dfh3.preds.apply(lambda x: 0.5 if x > threshold else 0)
dfh3["thresh_fill_color"] = dfh3["preds_thresh"].apply(lambda x: [255, int(255 * (1 - x)), int(255 * (1 - x)), 100])

In [425]:
view_state_grid = pdk.ViewState(
    latitude=34, 
    longitude=-117, 
    zoom=6, 
    bearing=0, 
    pitch=0
)
layer_grid = pdk.Layer(
    "H3HexagonLayer",
    dfh3.reset_index().to_dict("records"),
    pickable=True,
    stroked=False,
    filled=True,
    extruded=False,
    get_hexagon="h3_04",
    get_fill_color="thresh_fill_color",
    line_width_min_pixels=0,
)
r_grid = pdk.Deck(
    layers=[layer_grid], 
    initial_view_state=view_state_grid,
    map_style=pdk.map_styles.LIGHT,
)
#r_grid.show()
r_grid.to_html("crotalus_ruber_r4_from_r5_agg_max_thresh.html", notebook_display=False)